# 차선 분할 모델 학습 (Colab)

런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 위에서부터 차례로 실행.

데이터: `pack_dataset.py`로 만든 `lane_dataset_part0.zip`, `part1.zip`, ... 을 Google Drive의 `autorace/` 폴더에 올려 둔다.

모델은 2클래스(노란 차선 / 흰 표시)다. 로그의 val IoU는 두 클래스 평균, 괄호 안이 클래스별 값.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 레포 받기
비공개 레포라 GitHub 토큰이 필요하다 (GitHub → Settings → Developer settings → Fine-grained tokens, 이 레포 Contents: Read-only). 토큰은 노트북에 저장되지 않는다.

In [ ]:
from getpass import getpass
token = getpass('GitHub token: ')
!git clone -q https://{token}@github.com/ryanpaek/Autorace2026_KUUVe.git /content/repo
del token
!ls /content/repo

## 2. 데이터 (Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Drive의 autorace/ 폴더에 있는 lane_dataset*.zip 을 전부 같은 곳에 푼다 (part0, part1, ... 합쳐짐)
!mkdir -p /content/data && for f in /content/drive/MyDrive/autorace/lane_dataset*.zip; do unzip -q -o "$f" -d /content/data; done
!find /content/data -maxdepth 2 -type d
# images/, masks/ 가 들어 있는 폴더를 DATA로 지정
import glob, os
DATA = os.path.dirname(glob.glob('/content/data/**/images', recursive=True)[0])
print('DATA =', DATA, len(os.listdir(DATA + '/images')), 'images')

## 3. 학습
Colab에는 torch가 이미 있다. 메모리가 부족하면 `--batch 4`.

In [ ]:
!pip -q install onnx
!cd /content/repo/training/lane_seg && python train_lane_seg.py --data {DATA} --out /content/run --pretrained --epochs 60 --batch 8

## 4. 결과 확인 (초록 맞춤, 빨강 놓침, 파랑 오검출)

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('/content/run/val_preview/*'))[:6]:
    display(Image(f, width=900))

## 5. Drive에 저장 → LIMO로 복사
`lane_seg.onnx`를 LIMO에 옮기고 params.yaml에서 `lane_mask_source: model`, `seg_model_path` 지정.

In [ ]:
import datetime, shutil
tag = datetime.datetime.now().strftime('%m%d_%H%M')
dst = f'/content/drive/MyDrive/autorace/models/{tag}'
shutil.copytree('/content/run', dst)
print('saved to', dst)